# EthicCheck AI - Lab 3 data pipeline

**Pathway A (TXT)** expands **C4, versioned knowledge access**, from the Week 2 architecture. FR4 needs a guideline reference, RAI1 needs verifiable provenance, and NFR5 needs a repeatable refresh. The downstream E4 citation audit and E12 update test will use this evidence. This notebook prepares the retrieval corpus. It uses no model or API key.

Open this notebook in Google Colab and choose **Runtime → Run all**. On Colab, the first cell creates a working directory under `/content/ethiccheck-week3` and retrieves the two source files from the project repository when they are missing. For offline use, extract or clone the repository and run with its root as the working directory. For a new source version, review the changes and deliberately update the recorded fingerprints. The last cell creates a ZIP of the generated data in Colab's Files panel.

In [1]:
# Part A: workspace and exact input selection
from collections import Counter
from datetime import datetime, timezone
from hashlib import sha256
from pathlib import Path
from tempfile import TemporaryDirectory
from urllib.request import urlopen
import json
import platform
import re
import shutil

COLAB = Path('/content').is_dir()
BASE = Path('/content/ethiccheck-week3') if COLAB else Path.cwd()
RAW = BASE / 'data/sources'
RAW.mkdir(parents=True, exist_ok=True)
REPOSITORY_RAW = 'https://raw.githubusercontent.com/nafizshuk/ai-engineering-ethiccheck-ai/main'
EXPECTED_HASHES = {'data_and_fairness.txt': '1b4a900c3e22f3c751d97a96a50aad9e7a4b6c903559d6c95cd5e6d2243d16e0', 'risk_and_oversight.txt': '759c3d4492c46b4caaab8724f90799400bb697016ebc07e3feda39ea3e03ce25'}
NIST = 'https://doi.org/10.6028/NIST.AI.100-1'
SOURCE_META = {
    'data_and_fairness.txt': {'source_id':'S01', 'title':'Data and fairness', 'criteria':['PRI-01', 'FAI-01']},
    'risk_and_oversight.txt': {'source_id':'S02', 'title':'Risk and oversight', 'criteria':['SAF-01','ACC-01','EXP-01']},
}
NIST_SECTIONS = {
    'SAF-01':'3.2', 'ACC-01':'3.4', 'EXP-01':'3.5', 'PRI-01':'3.6', 'FAI-01':'3.7',
}
VERSION = '1.0'
LIMIT = 520
METHOD = 'heading-and-paragraph-aware, word fallback, no overlap'
RIGHTS = 'Original EthicCheck course material written for this repo. NIST references are attributed and linked.'
HEADER = re.compile(r'^## ([A-Z]+-\d+) \| (.+)$')

def utc_now():
    return datetime.now(timezone.utc).isoformat(timespec='seconds')

def fingerprint(payload):
    return {'bytes':len(payload), 'sha256':sha256(payload).hexdigest()}

def read_inputs(allow_download=False):
    payloads = {}
    for filename, expected in EXPECTED_HASHES.items():
        path = RAW / filename
        if not path.is_file() and allow_download:
            with urlopen(f'{REPOSITORY_RAW}/data/sources/{filename}', timeout=25) as response:
                candidate = response.read(100_001)
            if sha256(candidate).hexdigest() != expected:
                raise ValueError(f'Download does not match approved fingerprint: {filename}')
            path.write_bytes(candidate)
        if not path.is_file():
            raise FileNotFoundError(f'Place {filename} in {RAW}')
        payload = path.read_bytes()
        if sha256(payload).hexdigest() != expected:
            raise ValueError(f'{filename} changed. Review the new version and update its approved fingerprint.')
        payloads[filename] = payload
    return payloads

inputs = read_inputs(allow_download=COLAB)
for name, data in inputs.items():
    print(name, fingerprint(data))

data_and_fairness.txt {'bytes': 1170, 'sha256': '1b4a900c3e22f3c751d97a96a50aad9e7a4b6c903559d6c95cd5e6d2243d16e0'}
risk_and_oversight.txt {'bytes': 1644, 'sha256': '759c3d4492c46b4caaab8724f90799400bb697016ebc07e3feda39ea3e03ce25'}


In [2]:
# A3: inspect both files and extract named sections with line locations
def inspect_sources(payloads):
    sections, observations = [], []
    for name, blob in payloads.items():
        source = SOURCE_META[name]
        raw = blob.decode('utf-8-sig')
        lines = raw.splitlines()
        headers = [(i, HEADER.fullmatch(line)) for i, line in enumerate(lines)
                   if line.startswith('## ')]
        ids = [match[1] for _, match in headers if match is not None]
        if any(match is None for _, match in headers) or ids != source['criteria']:
            raise ValueError(f'Wrong or malformed headings in {name}: {ids}')
        if not headers or any(line.strip() for line in lines[:headers[0][0]]):
            raise ValueError(f'Unassigned text or no sections in {name}')
        for index, (heading, match) in enumerate(headers):
            stop = headers[index + 1][0] if index + 1 < len(headers) else len(lines)
            paragraphs, text_lines, line_map = [], [], []
            for pos in range(heading + 1, stop):
                cleaned = ' '.join(lines[pos].split())
                if not cleaned:
                    if text_lines:
                        paragraphs.append((' '.join(text_lines), line_map[0], line_map[-1]))
                        text_lines, line_map = [], []
                    continue
                text_lines.append(cleaned)
                line_map.append(pos + 1)
            if text_lines:
                paragraphs.append((' '.join(text_lines), line_map[0], line_map[-1]))
            if not paragraphs:
                raise ValueError(f'Empty section {match[1]} in {name}')
            sections.append({
                'filename':name, 'source_id':source['source_id'], 'criterion_id':match[1],
                'category':match[2], 'paragraphs':paragraphs, 'heading_line':heading + 1,
                'source_sha256':fingerprint(blob)['sha256'],
            })
        observations.append({
            **source, 'filename':name, 'physical_lines':len(lines),
            'bom':blob.startswith(b'\xef\xbb\xbf'), 'characters':len(raw),
            'paragraph_count':sum(len(s['paragraphs']) for s in sections if s['filename']==name),
            **fingerprint(blob),
        })
    return sections, observations

sections, observations = inspect_sources(inputs)
for item in observations:
    print(item['source_id'], item['filename'], '|', item['physical_lines'], 'lines |',
          item['paragraph_count'], 'paragraphs | heading IDs:', ', '.join(item['criteria']))
    sample = next(section for section in sections if section['filename'] == item['filename'])
    body, lo, hi = sample['paragraphs'][0]
    print(f"  {sample['criterion_id']} lines {lo}-{hi}: {body[:140]}...")
print('Normalisation: UTF-8 BOM and line endings handled. Blank lines mark paragraph boundaries.')

S01 data_and_fairness.txt | 11 lines | 4 paragraphs | heading IDs: PRI-01, FAI-01
  PRI-01 lines 3-3: Find the report passages that describe what information enters the application, why it is needed, and who can access it. Inspect how long or...
S02 risk_and_oversight.txt | 17 lines | 6 paragraphs | heading IDs: SAF-01, ACC-01, EXP-01
  SAF-01 lines 3-3: Identify one way the proposed AI feature could give an incorrect or inappropriate result in ordinary use. Look for a description of who migh...
Normalisation: UTF-8 BOM and line endings handled. Blank lines mark paragraph boundaries.


## Part B - Compare two approaches on the same text

Both methods process identical normalised criterion text and never cross a criterion heading. The baseline cuts at 520 characters. The selected method groups whole paragraphs where they fit. A paragraph that exceeds the limit splits at the last available space. Exported offsets refer to each criterion's normalised text, while line ranges refer to the original TXT file. TSV/CSV-style processing is unnecessary for the document inputs.

In [3]:
# B2-B3: prepare, compare and validate Pathway A
def normalized(section):
    text, ranges = '', []
    for paragraph, first, last in section['paragraphs']:
        if text:
            text += '\n\n'
        start = len(text)
        text += paragraph
        ranges.append((start, len(text), first, last))
    return text, ranges

def spans_for(text, ranges, method):
    if method == 'fixed':
        return [(start, min(start + LIMIT, len(text))) for start in range(0, len(text), LIMIT)]
    spans = []
    start = 0
    while start < len(text):
        cap = min(start + LIMIT, len(text))
        # Prefer an end that also retains the paragraph separator in this span.
        ends = [ranges[index + 1][0] if index + 1 < len(ranges) else end
                for index, (_, end, _, _) in enumerate(ranges)]
        candidates = [end for end in ends if start < end <= cap]
        if candidates:
            end = max(candidates)
        else:
            prior = max(text.rfind(' ', start, cap), text.rfind('\n', start, cap))
            if prior < start:
                raise ValueError('A token exceeds the character limit')
            end = prior + 1
        spans.append((start, end))
        start = end
    return spans

def prepare(sections, method):
    records, comparison = [], {}
    for section in sections:
        text, ranges = normalized(section)
        spans = spans_for(text, ranges, method)
        boundaries = [end for _, end in spans[:-1]]
        comparison[section['criterion_id']] = {
            'records':len(spans),
            'word_splits':sum(text[b-1].isalnum() and text[b].isalnum() for b in boundaries),
            'within_paragraph_cuts':sum(any(lo < b < hi for lo, hi, _, _ in ranges) for b in boundaries),
        }
        for index, (start, end) in enumerate(spans, 1):
            touched = [(first,last) for lo,hi,first,last in ranges if lo < end and start < hi]
            records.append({
                'record_id':f"{section['source_id']}-{section['criterion_id']}-{index:02d}",
                'source_id':section['source_id'], 'source':f"data/sources/{section['filename']}",
                'source_version':VERSION, 'source_sha256':section['source_sha256'],
                'criterion_id':section['criterion_id'], 'category':section['category'],
                'heading_line':section['heading_line'], 'line_start':touched[0][0],
                'line_end':touched[-1][1], 'char_start':start, 'char_end':end, 'text':text[start:end],
                'reference_title':'NIST AI RMF 1.0', 'reference_version':'NIST AI 100-1, January 2023',
                'reference_section':NIST_SECTIONS[section['criterion_id']], 'reference_url':NIST,
                'processing':METHOD if method == 'paragraph' else 'fixed characters',
            })
    return records, comparison

def verify_records(records, sections):
    if not records or len({row['record_id'] for row in records}) != len(records):
        raise ValueError('Empty output or duplicate IDs')
    if {r['criterion_id'] for r in records} != {s['criterion_id'] for s in sections}:
        raise ValueError('Coverage mismatch')
    for section in sections:
        expected, ranges = normalized(section)
        rows = [row for row in records if row['criterion_id'] == section['criterion_id']]
        cursor = 0
        for row in rows:
            start, end = row['char_start'], row['char_end']
            touched = [(first,last) for lo,hi,first,last in ranges if lo < end and start < hi]
            if start != cursor or not (0 < end-start <= LIMIT) or not row['text'].strip():
                raise ValueError(f"Invalid boundary or empty text: {row['record_id']}")
            if (row['text'] != expected[start:end] or row['source_id'] != section['source_id']
                or row['source'] != f"data/sources/{section['filename']}"
                or row['source_sha256'] != section['source_sha256']
                or row['source_version'] != VERSION or row['heading_line'] != section['heading_line']
                or (row['line_start'], row['line_end']) != (touched[0][0], touched[-1][1])
                or row['reference_section'] != NIST_SECTIONS[section['criterion_id']]
                or row['reference_url'] != NIST):
                raise ValueError(f"Broken provenance: {row['record_id']}")
            cursor = end
        if cursor != len(expected):
            raise ValueError(f"Lost or unassigned text: {section['criterion_id']}")

def write_json(path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_bytes((json.dumps(payload, ensure_ascii=False, indent=2) + '\n').encode('utf-8'))

def export_records(path, records, sections):
    verify_records(records, sections)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_bytes(''.join(json.dumps(record, ensure_ascii=False) + '\n' for record in records).encode('utf-8'))
    blob = path.read_bytes()
    reloaded = [json.loads(line) for line in blob.decode('utf-8').splitlines()]
    verify_records(reloaded, sections)
    if reloaded != records:
        raise ValueError('Export readback changed records')
    return {**fingerprint(blob), 'records':len(reloaded)}

fixed, fixed_stats = prepare(sections, 'fixed')
chosen, chosen_stats = prepare(sections, 'paragraph')
verify_records(fixed, sections)
part_b = BASE / 'data/processed/prepared_text.jsonl'
part_b_evidence = export_records(part_b, chosen, sections)
for criterion in fixed_stats:
    print(criterion, 'fixed:', fixed_stats[criterion], '| paragraph:', chosen_stats[criterion])
print('Export validation PASS:', part_b.relative_to(BASE).as_posix(), part_b_evidence)
sample = next(record for record in chosen if record['criterion_id'] == 'PRI-01')
print(f"Representative record: {sample['record_id']} | lines {sample['line_start']}-{sample['line_end']} | NIST §{sample['reference_section']}")
print(repr(sample['text'].strip()[:180]))

PRI-01 fixed: {'records': 2, 'word_splits': 0, 'within_paragraph_cuts': 1} | paragraph: {'records': 2, 'word_splits': 0, 'within_paragraph_cuts': 0}
FAI-01 fixed: {'records': 2, 'word_splits': 1, 'within_paragraph_cuts': 1} | paragraph: {'records': 2, 'word_splits': 0, 'within_paragraph_cuts': 0}
SAF-01 fixed: {'records': 2, 'word_splits': 1, 'within_paragraph_cuts': 1} | paragraph: {'records': 2, 'word_splits': 0, 'within_paragraph_cuts': 0}
ACC-01 fixed: {'records': 1, 'word_splits': 0, 'within_paragraph_cuts': 0} | paragraph: {'records': 1, 'word_splits': 0, 'within_paragraph_cuts': 0}
EXP-01 fixed: {'records': 1, 'word_splits': 0, 'within_paragraph_cuts': 0} | paragraph: {'records': 1, 'word_splits': 0, 'within_paragraph_cuts': 0}
Export validation PASS: data/processed/prepared_text.jsonl {'bytes': 7549, 'sha256': '960ea4ccb1a529328242101f4228d7b765303981c1b5030b2dddaed83967123c', 'records': 8}
Representative record: S01-PRI-01-01 | lines 3-3 | NIST §3.6
'Find the report passages t

In [4]:
# B4: register source -> prepared output mappings and verify saved manifest
def source_lineage(observations, records, output_name, output_evidence):
    counts = Counter(row['source_id'] for row in records)
    return [{
        **item, 'source_path':f"data/sources/{item['filename']}", 'source_version':VERSION,
        'owner':'EthicCheck AI project', 'access_rights':RIGHTS,
        'reference_title':'NIST AI RMF 1.0', 'reference_url':NIST,
        'reference_sections':{cid:NIST_SECTIONS[cid] for cid in item['criteria']},
        'prepared_output':output_name, 'prepared_sha256':output_evidence['sha256'],
        'prepared_bytes':output_evidence['bytes'], 'record_count':counts[item['source_id']],
        'processing_note':f'{METHOD}, {LIMIT} characters per chunk, physical TXT line ranges',
    } for item in observations]

def verify_manifest(path, records, output_path):
    manifest = json.loads(path.read_text(encoding='utf-8'))
    if manifest['records'] != records:
        raise ValueError('Manifest differs from the registered mappings')
    output_evidence = fingerprint(output_path.read_bytes())
    for row in manifest['records']:
        source = fingerprint((BASE / row['source_path']).read_bytes())
        if source['sha256'] != row['sha256'] or source['bytes'] != row['bytes']:
            raise ValueError('Source fingerprint mismatch')
        if output_evidence != {'sha256':row['prepared_sha256'], 'bytes':row['prepared_bytes']}:
            raise ValueError('Output fingerprint mismatch')
        if not row['record_count']:
            raise ValueError('Source has no output mapping')

mapped = source_lineage(observations, chosen, 'data/processed/prepared_text.jsonl', part_b_evidence)
part_b_manifest = BASE / 'data/week3_manifest.json'
write_json(part_b_manifest, {'version':1, 'created_utc':utc_now(), 'pathway':'A - TXT',
                            'quality_status':'Format and provenance checks passed. Topic 4 review pending.',
                            'comparison':{'fixed':fixed_stats,'paragraph':chosen_stats}, 'records':mapped})
verify_manifest(part_b_manifest, mapped, part_b)
print('Lineage validation PASS:', len(mapped), 'sources; data/week3_manifest.json')

Lineage validation PASS: 2 sources; data/week3_manifest.json


In [5]:
# Part C: execute integrated pipeline and rerun from the same approved inputs
OUTPUT_NAME = 'data/pipeline_v1/text/prepared_text.jsonl'

def execute_pipeline(folder):
    fresh_inputs = read_inputs()
    parsed, inspected = inspect_sources(fresh_inputs)
    records, stats = prepare(parsed, 'paragraph')
    path = folder / 'prepared_text.jsonl'
    evidence = export_records(path, records, parsed)
    lineage = source_lineage(inspected, records, OUTPUT_NAME, evidence)
    return {
        'pathway':'A - TXT', 'created_utc':utc_now(), 'python':platform.python_version(),
        'workflow':'batch ETL', 'processing':{'method':METHOD, 'maximum_characters':LIMIT, 'overlap':0},
        'sources':{name:fingerprint(data) for name,data in fresh_inputs.items()},
        'validation':{'file':OUTPUT_NAME, **evidence},
        'comparison_metrics':stats, 'records':lineage,
        'notes':'Application-authored guides link to NIST sections. TXT line locations are not NIST page numbers.',
    }

final_folder = BASE / 'data/pipeline_v1/text'
# Use temporary staging for both runs so failed checks cannot replace an approved export.
with TemporaryDirectory() as scratch:
    job = Path(scratch)
    first = execute_pipeline(job / 'first')
    second = execute_pipeline(job / 'second')
    compared = ['processing','sources','validation','comparison_metrics','records']
    if any(first[key] != second[key] for key in compared):
        raise ValueError('Repeatability failed: compare inputs, settings, bytes, counts and lineage')
    if first['validation']['sha256'] != part_b_evidence['sha256']:
        raise ValueError('Part C output differs from the chosen Part B output')
    first['repeatability'] = {'status':'PASS', 'runs':2, 'matched_fields':compared,
                              'rerun_finished_utc':second['created_utc'],
                              'part_b_output_equal':True}
    final_folder.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(job / 'first/prepared_text.jsonl', final_folder / 'prepared_text.jsonl')
    manifest_path = final_folder / 'pipeline_manifest.json'
    write_json(manifest_path, first)
    verify_manifest(manifest_path, first['records'], final_folder / 'prepared_text.jsonl')
print('Repeatability PASS: both runs have the same input hashes, output count, byte size, SHA-256 and mappings.')
print('Part B and Part C exports are byte-identical.')
print('Integrated output:', first['validation'])
print('Saved:', manifest_path.relative_to(BASE).as_posix())

Repeatability PASS: both runs have the same input hashes, output count, byte size, SHA-256 and mappings.
Part B and Part C exports are byte-identical.
Integrated output: {'file': 'data/pipeline_v1/text/prepared_text.jsonl', 'bytes': 7549, 'sha256': '960ea4ccb1a529328242101f4228d7b765303981c1b5030b2dddaed83967123c', 'records': 8}
Saved: data/pipeline_v1/text/pipeline_manifest.json


In [6]:
# Colab: package generated data before closing the temporary runtime.
if COLAB:
    from zipfile import ZipFile, ZIP_DEFLATED
    bundle = BASE / 'week3-generated-data.zip'
    with ZipFile(bundle, 'w', ZIP_DEFLATED) as archive:
        for item in (part_b, part_b_manifest, final_folder/'prepared_text.jsonl', manifest_path):
            archive.write(item, item.relative_to(BASE).as_posix())
    print('Download from the Colab Files panel:', bundle)
else:
    print('Generated artefacts are under data/processed and data/pipeline_v1/text.')

Generated artefacts are under data/processed and data/pipeline_v1/text.
